In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
import os
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)

catalog_name = "my_catalog"

metadata_rows = []

# Get all schemas
schemas = spark.sql(f"""
SELECT schema_name
FROM {catalog_name}.information_schema.schemata
""").collect()

for schema in schemas:

    schema_name = schema["schema_name"]

    print(f"Processing schema: {schema_name}")

    # Get tables
    tables = spark.sql(f"""
    SELECT
        table_catalog,
        table_schema,
        table_name,
        table_type
    FROM {catalog_name}.information_schema.tables
    WHERE table_schema = '{schema_name}'
    """)

    for table in tables.collect():

        table_name = table["table_name"]

        try:

            # Table details
            table_detail = spark.sql(f"""
            DESCRIBE DETAIL
            {catalog_name}.{schema_name}.{table_name}
            """).first()

            table_comment = None

            try:
                table_comment = spark.sql(f"""
                DESCRIBE TABLE EXTENDED
                {catalog_name}.{schema_name}.{table_name}
                """).filter("col_name='Comment'").first()

                if table_comment:
                    table_comment = table_comment["data_type"]

            except:
                pass

            # Column metadata
            columns = spark.sql(f"""
            SELECT
                column_name,
                data_type,
                is_nullable,
                comment
            FROM {catalog_name}.information_schema.columns
            WHERE table_schema = '{schema_name}'
            AND table_name = '{table_name}'
            ORDER BY ordinal_position
            """)

            for col in columns.collect():

                metadata_rows.append(
                    Row(
                        catalog_name=table["table_catalog"],
                        schema_name=schema_name,
                        table_name=table_name,
                        table_type=table["table_type"],
                        column_name=col["column_name"],
                        data_type=col["data_type"],
                        nullable=col["is_nullable"],
                        column_comment=col["comment"],
                        business_description=None,
                        sensitivity=None,
                        primary_key=None,
                        foreign_key=None,
                        table_owner=None,
                        table_comment=table_comment,
                        column_tags=None,
                        table_tags=None,
                        location=table_detail["location"],
                        owner=table_detail["properties"].get("owner", None)
                                if table_detail["properties"] else None,
                        num_files=table_detail["numFiles"]
                    )
                )

        except Exception as e:

            print(
                f"Failed processing "
                f"{catalog_name}.{schema_name}.{table_name}: {e}"
            )

# Create DataFrame
metadata_schema = StructType([
    StructField("catalog_name", StringType(), True),
    StructField("schema_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("table_type", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("nullable", StringType(), True),
    StructField("column_comment", StringType(), True),
    StructField("business_description", StringType(), True),
    StructField("sensitivity", StringType(), True),
    StructField("primary_key", StringType(), True),
    StructField("foreign_key", StringType(), True),
    StructField("table_owner", StringType(), True),
    StructField("table_comment", StringType(), True),
    StructField("column_tags", StringType(), True),
    StructField("table_tags", StringType(), True),
    StructField("location", StringType(), True),
    StructField("owner", StringType(), True),
    StructField("num_files", LongType(), True),
])

metadata_df = spark.createDataFrame(metadata_rows, schema=metadata_schema)

metadata_df = metadata_df.withColumn(
    "metadata_extract_timestamp",
    current_timestamp()
)

display(metadata_df)

# Output folder
output_folder = "/Volumes/my_catalog/landing/raw_files/data_dictionary"

os.makedirs(output_folder, exist_ok=True)

# Convert to Pandas
pdf = metadata_df.toPandas()

# Single CSV file
csv_file = "/Volumes/my_catalog/landing/raw_files/data_dictionary/my_catalog_metadata.csv"

pdf.to_csv(
    csv_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV created: {csv_file}")